In [2]:
# Imports
import pandas as pd
import numpy as np

In [3]:
# Load raw data
trans = pd.read_csv('../data/raw/train_transaction.csv')
ident = pd.read_csv('../data/raw/train_identity.csv')

In [4]:
# First look
print("=== TRANSACTION ===")
print(trans.shape)
print(trans.dtypes.value_counts())
print(trans.isnull().mean().sort_values(ascending=False).head(20))

print("\n=== IDENTITY ===")
print(ident.shape)
print(ident.dtypes.value_counts())
print(ident.isnull().mean().sort_values(ascending=False).head(20))

print("\n=== TARGET ===")
print(trans['isFraud'].value_counts(normalize=True))

=== TRANSACTION ===
(590540, 394)
float64    376
str         14
int64        4
Name: count, dtype: int64
dist2    0.936284
D7       0.934099
D13      0.895093
D14      0.894695
D12      0.890410
D6       0.876068
D9       0.873123
D8       0.873123
V153     0.861237
V139     0.861237
V162     0.861237
V161     0.861237
V154     0.861237
V138     0.861237
V158     0.861237
V157     0.861237
V163     0.861237
V156     0.861237
V155     0.861237
V149     0.861237
dtype: float64

=== IDENTITY ===
(144233, 41)
float64    23
str        17
int64       1
Name: count, dtype: int64
id_24         0.967088
id_25         0.964419
id_07         0.964259
id_08         0.964259
id_21         0.964231
id_26         0.964204
id_23         0.964162
id_27         0.964162
id_22         0.964162
id_18         0.687221
id_03         0.540161
id_04         0.540161
id_33         0.491871
id_09         0.480521
id_10         0.480521
id_30         0.462224
id_32         0.462079
id_34         0.460560
id_14  

In [5]:
# Identity join and column overview
print("=== IDENTITY ===")
print(ident.shape)
print(ident.columns.tolist())

print("\n=== TRANSACTION COLUMNS ===")
print(trans.columns.tolist())

=== IDENTITY ===
(144233, 41)
['TransactionID', 'id_01', 'id_02', 'id_03', 'id_04', 'id_05', 'id_06', 'id_07', 'id_08', 'id_09', 'id_10', 'id_11', 'id_12', 'id_13', 'id_14', 'id_15', 'id_16', 'id_17', 'id_18', 'id_19', 'id_20', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_32', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']

=== TRANSACTION COLUMNS ===
['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6', 'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain', 'R_emaildomain', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V1

In [7]:
# Merge and join quality check
df = trans.merge(ident, on='TransactionID', how='left')

print(f"Shape after join: {df.shape}")

df['has_identity'] = df['id_01'].notna()
print("\nFraud rate by identity availability:")
print(df.groupby('has_identity')['isFraud'].mean().round(4))

print("\nClass balance check:")
print(df['isFraud'].value_counts(normalize=True).round(4))
df = df.copy()

Shape after join: (590540, 434)

Fraud rate by identity availability:
has_identity
False    0.0209
True     0.0785
Name: isFraud, dtype: float64

Class balance check:
isFraud
0    0.965
1    0.035
Name: proportion, dtype: float64


/var/folders/wn/b78qd5gn3pd2clt20k9z6bjr0000gn/T/ipykernel_2652/1526260663.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['has_identity'] = df['id_01'].notna()


In [8]:
# Temporal distribution
import matplotlib.pyplot as plt

print(f"TransactionDT min: {df['TransactionDT'].min():,}")
print(f"TransactionDT max: {df['TransactionDT'].max():,}")
print(f"Range in days: {(df['TransactionDT'].max() - df['TransactionDT'].min()) / 86400:.0f}")

# Split points
total = len(df)
train_end = int(total * 0.70)
val_end   = int(total * 0.85)

df_sorted = df.sort_values('TransactionDT')
train_cutoff = df_sorted.iloc[train_end]['TransactionDT']
val_cutoff   = df_sorted.iloc[val_end]['TransactionDT']

print(f"\nTrain cutoff DT:      {train_cutoff:,}")
print(f"Validation cutoff DT: {val_cutoff:,}")
print(f"\nTrain fraud rate:      {df_sorted.iloc[:train_end]['isFraud'].mean():.4f}")
print(f"Validation fraud rate: {df_sorted.iloc[train_end:val_end]['isFraud'].mean():.4f}")
print(f"Test fraud rate:       {df_sorted.iloc[val_end:]['isFraud'].mean():.4f}")

TransactionDT min: 86,400
TransactionDT max: 15,811,131
Range in days: 182

Train cutoff DT:      10,438,003
Validation cutoff DT: 13,151,880

Train fraud rate:      0.0352
Validation fraud rate: 0.0343
Test fraud rate:       0.0348


Train:      TransactionDT < 10,438,003   (~413,000 rows)

Validation: 10,438,003 ≤ DT < 13,151,880 (~89,000 rows)

Test:       TransactionDT ≥ 13,151,880   (~89,000 rows)

## EDA Findings : Data Split

- Dataset spans **182 days** (TransactionDT: 86,400 → 15,811,131)
- Time-based 70/15/15 split applied ; no random shuffle
- Train cutoff: 10,438,003 | Validation cutoff: 13,151,880
- Fraud rate is stable across splits (3.52% / 3.43% / 3.48%) ; no concept drift detected
- Transactions with identity data have **3.75x higher fraud rate** (7.85% vs 2.09%)
  → `has_identity` flag is a strong feature candidate

In [9]:
# TransactionAmt distribution
print("=== TRANSACTION AMOUNT ===")
print(df['TransactionAmt'].describe().round(2))
print(f"\nSkewness: {df['TransactionAmt'].skew():.2f}")
print(f"% rounded amounts: {(df['TransactionAmt'] % 1 == 0).mean():.4f}")
print(f"\nFraud rate for rounded amounts: {df[df['TransactionAmt'] % 1 == 0]['isFraud'].mean():.4f}")
print(f"Fraud rate for non-rounded amounts: {df[df['TransactionAmt'] % 1 != 0]['isFraud'].mean():.4f}")

=== TRANSACTION AMOUNT ===
count    590540.00
mean        135.03
std         239.16
min           0.25
25%          43.32
50%          68.77
75%         125.00
max       31937.39
Name: TransactionAmt, dtype: float64

Skewness: 14.37
% rounded amounts: 0.5165

Fraud rate for rounded amounts: 0.0357
Fraud rate for non-rounded amounts: 0.0343


## EDA Findings — Transaction Amount

- Amount is heavily right-skewed (skewness: 14.37) — log transform confirmed necessary
- Median $68.77 vs mean $135.03 — long tail of high-value transactions
- 51.6% of transactions use rounded amounts — weak fraud signal (3.57% vs 3.43%)
- `amt_log`, `amt_rounded`, `amt_cents` features confirmed for pipeline

In [10]:
# Email domain analysis
print("=== PURCHASER EMAIL DOMAIN ===")
print(f"Unique domains: {df['P_emaildomain'].nunique()}")
print(f"Missing: {df['P_emaildomain'].isna().mean():.4f}")
print("\nTop 10 domains:")
print(df['P_emaildomain'].value_counts().head(10))

print("\n=== FRAUD RATE BY TOP PURCHASER DOMAINS ===")
top_domains = df['P_emaildomain'].value_counts().head(10).index
print(df[df['P_emaildomain'].isin(top_domains)]
      .groupby('P_emaildomain')['isFraud']
      .agg(['mean','count'])
      .sort_values('mean', ascending=False)
      .round(4))

=== PURCHASER EMAIL DOMAIN ===
Unique domains: 59
Missing: 0.1599

Top 10 domains:
P_emaildomain
gmail.com        228355
yahoo.com        100934
hotmail.com       45250
anonymous.com     36998
aol.com           28289
comcast.net        7888
icloud.com         6267
outlook.com        5096
msn.com            4092
att.net            4033
Name: count, dtype: int64

=== FRAUD RATE BY TOP PURCHASER DOMAINS ===
                 mean   count
P_emaildomain                
outlook.com    0.0946    5096
hotmail.com    0.0530   45250
gmail.com      0.0435  228355
icloud.com     0.0314    6267
comcast.net    0.0312    7888
anonymous.com  0.0232   36998
yahoo.com      0.0228  100934
msn.com        0.0220    4092
aol.com        0.0218   28289
att.net        0.0074    4033


In [11]:
print(df[df['P_emaildomain'] == 'anonymous.com']['isFraud'].mean())

0.02321747121466025


- anonymous.com (4th most common, 36,998 transactions) — fraud rate 2.32%, 
  below average despite name suggesting anonymity

In [12]:
# C features analysis
c_cols = [f'C{i}' for i in range(1, 15)]

print("=== C FEATURES — MISSING RATES ===")
print(df[c_cols].isna().mean().round(4))

print("\n=== C FEATURES — FRAUD CORRELATION ===")
correlations = df[c_cols + ['isFraud']].corr()['isFraud'].drop('isFraud')
print(correlations.sort_values(ascending=False).round(4))

=== C FEATURES — MISSING RATES ===
C1     0.0
C2     0.0
C3     0.0
C4     0.0
C5     0.0
C6     0.0
C7     0.0
C8     0.0
C9     0.0
C10    0.0
C11    0.0
C12    0.0
C13    0.0
C14    0.0
dtype: float64

=== C FEATURES — FRAUD CORRELATION ===
C2     0.0372
C8     0.0321
C12    0.0319
C1     0.0306
C4     0.0304
C10    0.0284
C7     0.0282
C11    0.0275
C6     0.0209
C14    0.0079
C3    -0.0068
C13   -0.0111
C5    -0.0308
C9    -0.0317
Name: isFraud, dtype: float64


## EDA Findings — C Features (Count Features)

- All 14 C features are complete : zero missing values
- Linear correlation with fraud is weak (max 0.037) but non-linear signal likely exists
- C2, C8, C12 show highest positive correlation
- C5, C9 show slight negative correlation
- All 14 kept ; feature importance from XGBoost will determine final selection

In [13]:
# D features analysis
d_cols = [f'D{i}' for i in range(1, 16)]

print("=== D FEATURES — MISSING RATES ===")
missing = df[d_cols].isna().mean().sort_values(ascending=False)
print(missing.round(4))

print("\n=== D FEATURES — FRAUD CORRELATION (non-null) ===")
correlations = df[d_cols + ['isFraud']].corr()['isFraud'].drop('isFraud')
print(correlations.sort_values(ascending=False).round(4))

=== D FEATURES — MISSING RATES ===
D7     0.9341
D13    0.8951
D14    0.8947
D12    0.8904
D6     0.8761
D8     0.8731
D9     0.8731
D5     0.5247
D2     0.4755
D11    0.4729
D3     0.4451
D4     0.2860
D15    0.1509
D10    0.1287
D1     0.0021
dtype: float64

=== D FEATURES — FRAUD CORRELATION (non-null) ===
D14   -0.0087
D12   -0.0289
D9    -0.0443
D11   -0.0451
D3    -0.0463
D6    -0.0572
D13   -0.0594
D5    -0.0646
D1    -0.0672
D4    -0.0672
D10   -0.0720
D15   -0.0775
D2    -0.0836
D7    -0.1272
D8    -0.1426
Name: isFraud, dtype: float64


## EDA Findings — D Features (Time Delta Features)

- D7 (93.4%) through D6 (87.6%) are severely missing — missingness = no transaction history
- D1 is nearly complete (0.2% missing) — likely days since account creation
- D8 (-0.143) and D7 (-0.127) are strongest fraud signals found so far
- Negative correlation: longer time since last event = lower fraud risk
- Strategy confirmed: impute with -1 + add binary `is_missing` flag per D feature
- Do NOT drop high-missing D features — their missingness is the signal

In [14]:
# M features analysis
m_cols = [f'M{i}' for i in range(1, 10)]

print("=== M FEATURES — VALUE COUNTS ===")
for col in m_cols:
    vc = df[col].value_counts(dropna=False)
    fraud_by_val = df.groupby(df[col].fillna('NaN'))['isFraud'].mean().round(4)
    print(f"\n{col}: {dict(vc)}")
    print(f"  Fraud rate: {dict(fraud_by_val)}")

=== M FEATURES — VALUE COUNTS ===

M1: {'T': np.int64(319415), nan: np.int64(271100), 'F': np.int64(25)}
  Fraud rate: {'F': np.float64(0.0), 'NaN': np.float64(0.0528), 'T': np.float64(0.0199)}

M2: {'T': np.int64(285468), nan: np.int64(271100), 'F': np.int64(33972)}
  Fraud rate: {'F': np.float64(0.0349), 'NaN': np.float64(0.0528), 'T': np.float64(0.0181)}

M3: {nan: np.int64(271100), 'T': np.int64(251731), 'F': np.int64(67709)}
  Fraud rate: {'F': np.float64(0.0303), 'NaN': np.float64(0.0528), 'T': np.float64(0.0171)}

M4: {nan: np.int64(281444), 'M0': np.int64(196405), 'M2': np.int64(59865), 'M1': np.int64(52826)}
  Fraud rate: {'M0': np.float64(0.0366), 'M1': np.float64(0.0271), 'M2': np.float64(0.1137), 'NaN': np.float64(0.0186)}

M5: {nan: np.int64(350482), 'F': np.int64(132491), 'T': np.int64(107567)}
  Fraud rate: {'F': np.float64(0.0265), 'NaN': np.float64(0.0374), 'T': np.float64(0.0377)}

M6: {'F': np.int64(227856), 'T': np.int64(193324), nan: np.int64(169360)}
  Fraud rate:

## EDA Findings — M Features (Match Flags)

- M4 is categorical (M0/M1/M2/NaN) not binary — M2 has 11.37% fraud rate (3x average)
- NaN is consistently the highest-risk category across all M features
- NaN fraud rates: M1=5.28%, M6=7.07%, M7=4.58%, M8=4.58%
- M1 has only 25 F values — effectively binary (T vs NaN)
- Strategy confirmed: encode T=1, F=0, NaN=-1 for M1-M3, M5-M9
- M4 requires ordinal or one-hot encoding — not the standard T/F/NaN scheme

In [15]:
# Card features
print("=== CARD FEATURES — OVERVIEW ===")
for col in ['card1','card2','card3','card4','card5','card6']:
    n_unique = df[col].nunique()
    missing = df[col].isna().mean()
    fraud_var = df.groupby(df[col].fillna('missing'))['isFraud'].mean()
    print(f"{col}: {n_unique} unique | {missing:.4f} missing | "
          f"fraud rate range: {fraud_var.min():.4f}–{fraud_var.max():.4f}")

print("\n=== CARD4 & CARD6 VALUE COUNTS ===")
print(df['card4'].value_counts(dropna=False))
print()
print(df['card6'].value_counts(dropna=False))

=== CARD FEATURES — OVERVIEW ===
card1: 13553 unique | 0.0000 missing | fraud rate range: 0.0000–1.0000
card2: 500 unique | 0.0151 missing | fraud rate range: 0.0000–0.4068
card3: 114 unique | 0.0027 missing | fraud rate range: 0.0000–1.0000
card4: 4 unique | 0.0027 missing | fraud rate range: 0.0260–0.0773
card5: 119 unique | 0.0072 missing | fraud rate range: 0.0000–1.0000
card6: 4 unique | 0.0027 missing | fraud rate range: 0.0000–0.0668

=== CARD4 & CARD6 VALUE COUNTS ===
card4
visa                384767
mastercard          189217
american express      8328
discover              6651
NaN                   1577
Name: count, dtype: int64

card6
debit              439938
credit             148986
NaN                  1571
debit or credit        30
charge card            15
Name: count, dtype: int64


## EDA Findings — Card Features

- card1: 13,553 unique values — too high for direct encoding
  → use aggregation features (mean amt, std, count) computed on train only
- card4: 4 categories (visa/mastercard/amex/discover) — one-hot encode directly
  → amex highest fraud rate (7.73%), discover lowest (2.60%)
- card6: 4 categories (debit/credit/etc.) — one-hot encode directly
- card2, card3, card5: mid-cardinality anonymized — frequency encode
- All card fields: NaN treated as separate category

In [16]:
# V features summary
v_cols = [f'V{i}' for i in range(1, 340)]

missing_v = df[v_cols].isna().mean()
print(f"V features with >80% missing: {(missing_v > 0.8).sum()}")
print(f"V features with >50% missing: {(missing_v > 0.5).sum()}")
print(f"V features with <10% missing: {(missing_v < 0.1).sum()}")
print(f"\nMissing rate distribution:")
print(missing_v.describe().round(4))

# Identify groups by missing pattern
print(f"\nDistinct missing rate clusters (rounded to 2dp):")
print(missing_v.round(2).value_counts().sort_index())

V features with >80% missing: 47
V features with >50% missing: 159
V features with <10% missing: 86

Missing rate distribution:
count    339.0000
mean       0.4304
std        0.3588
min        0.0000
25%        0.0021
50%        0.4729
75%        0.7791
max        0.8612
dtype: float64

Distinct missing rate clusters (rounded to 2dp):
0.00    86
0.13    45
0.15    20
0.29    18
0.47    11
0.76    66
0.78    46
0.86    47
Name: count, dtype: int64


## EDA Findings — V Features (V1–V339)

- 339 Vesta-engineered features — intentionally anonymized
- 8 distinct missing-rate clusters identified: 0%, 13%, 15%, 29%, 47%, 76%, 78%, 86%
- Groups likely correspond to different transaction types in Vesta's system
- 86 features are always present (G1) — most reliable group
- 47 features are 86% missing (G8) — sparsest group, PCA likely best here
- Strategy: benchmark raw vs PCA per group in Phase 2
- Deep analysis deferred to 02_feature_engineering.ipynb

In [17]:
# Cell 13 — EDA Summary
print("=" * 55)
print("EDA COMPLETE — KEY FINDINGS SUMMARY")
print("=" * 55)

print(f"\nDataset: {df.shape[0]:,} transactions · {df.shape[1]} features")
print(f"Time span: 182 days")
print(f"Fraud rate: {df['isFraud'].mean():.4f} ({df['isFraud'].mean()*100:.2f}%)")

print("\n── Split ──────────────────────────────────────")
print(f"Train cutoff DT:      10,438,003")
print(f"Validation cutoff DT: 13,151,880")
print(f"Fraud rate stable across splits: 3.52% / 3.43% / 3.48%")

print("\n── Top signals found ──────────────────────────")
print(f"has_identity flag:    fraud 7.85% vs 2.09%  (3.75x)")
print(f"M4 = M2 category:     fraud 11.37%          (3.25x avg)")
print(f"D8 correlation:       -0.143 (strongest linear signal)")
print(f"outlook.com:          fraud 9.46%           (2.7x avg)")
print(f"card4 = amex:         fraud 7.73%           (2.2x avg)")

print("\n── Feature engineering confirmed ──────────────")
print(f"Log transform:        amt_log (skewness 14.37)")
print(f"Temporal features:    hour_of_day, day_of_week, is_weekend")
print(f"Email encoding:       frequency encode (59 unique domains)")
print(f"Card aggregations:    card1 mean/std/count (13,553 unique)")
print(f"M encoding:           T=1, F=0, NaN=-1 (NaN is high risk)")
print(f"M4 special encoding:  M0/M1/M2/NaN (M2=11.37% fraud)")
print(f"D strategy:           impute -1 + is_missing flag")
print(f"V strategy:           8 groups identified, benchmark in Ph2")
print(f"Identity flag:        has_identity binary feature")

EDA COMPLETE — KEY FINDINGS SUMMARY

Dataset: 590,540 transactions · 435 features
Time span: 182 days
Fraud rate: 0.0350 (3.50%)

── Split ──────────────────────────────────────
Train cutoff DT:      10,438,003
Validation cutoff DT: 13,151,880
Fraud rate stable across splits: 3.52% / 3.43% / 3.48%

── Top signals found ──────────────────────────
has_identity flag:    fraud 7.85% vs 2.09%  (3.75x)
M4 = M2 category:     fraud 11.37%          (3.25x avg)
D8 correlation:       -0.143 (strongest linear signal)
outlook.com:          fraud 9.46%           (2.7x avg)
card4 = amex:         fraud 7.73%           (2.2x avg)

── Feature engineering confirmed ──────────────
Log transform:        amt_log (skewness 14.37)
Temporal features:    hour_of_day, day_of_week, is_weekend
Email encoding:       frequency encode (59 unique domains)
Card aggregations:    card1 mean/std/count (13,553 unique)
M encoding:           T=1, F=0, NaN=-1 (NaN is high risk)
M4 special encoding:  M0/M1/M2/NaN (M2=11.37% fr